# Fine-Tuning LLaMA 3.1-8B with Unsloth
## Use Case: Legacy Netezza SQL → Optimized PySpark DataFrame Translation

**Target Hardware:** Google Colab Free Tier (Tesla T4, 15 GB VRAM)  
**Base Model:** `unsloth/Meta-Llama-3.1-8B-bnb-4bit`  
**Method:** QLoRA (4-bit quantization + LoRA adapters)

---

In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT SETUP
# Install Unsloth (Colab-optimised build) and all dependencies.
# Run once per runtime session; restart the kernel if prompted.
# ============================================================

# -- Unsloth (official Colab install command) -----------------
# Uses the pre-built CUDA wheel that matches the Colab runtime.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# -- Core training stack --------------------------------------
!pip install -q \
    transformers \
    "trl>=0.8.6" \
    peft \
    accelerate \
    bitsandbytes \
    datasets \
    xformers \
    triton

# -- Verify GPU availability ----------------------------------
import torch
assert torch.cuda.is_available(), "No GPU detected — change Runtime > Runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ============================================================
# CELL 2 — MODEL & LoRA SETUP
# Load the 4-bit quantised base model, then attach lightweight
# LoRA adapters to all projection layers for efficient fine-tuning.
# ============================================================

from unsloth import FastLanguageModel

# -----------------------------------------------------------
# Hyperparameters
# -----------------------------------------------------------
MAX_SEQ_LENGTH = 2048   # Max tokens per sample (fits T4 VRAM with batch=2)
LORA_RANK      = 16     # LoRA rank — higher = more capacity, more VRAM
LORA_ALPHA     = 16     # Scaling factor (keep equal to rank for clean initialisation)

# -----------------------------------------------------------
# 2a. Load the 4-bit quantised base model
# dtype=None lets Unsloth auto-detect the best dtype (float16 on T4).
# load_in_4bit=True applies bitsandbytes NF4 quantisation to cut
# the effective model footprint from ~16 GB to ~5 GB.
# -----------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,          # Auto: float16 on T4
    load_in_4bit    = True,
)

# -----------------------------------------------------------
# 2b. Attach LoRA adapters
# We target all 7 linear projection layers (attention + FFN)
# rather than attention-only, giving the model more capacity
# to learn the structural transformation from SQL to PySpark.
#
# use_gradient_checkpointing="unsloth" activates Unsloth's
# custom checkpointing which reduces VRAM by ~30% vs standard.
# -----------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r                        = LORA_RANK,
    lora_alpha               = LORA_ALPHA,
    lora_dropout             = 0,          # 0 is optimal for Unsloth
    target_modules           = [
        "q_proj", "k_proj", "v_proj", "o_proj",   # Attention layers
        "gate_proj", "up_proj", "down_proj",       # FFN layers
    ],
    bias                     = "none",
    use_gradient_checkpointing = "unsloth",
    random_state             = 42,
    use_rslora               = False,
    loftq_config             = None,
)

# -----------------------------------------------------------
# Sanity check: confirm trainable vs frozen parameter counts
# -----------------------------------------------------------
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters  : {total_params:,}")
print(f"Trainable (LoRA)  : {trainable_params:,}  ({100 * trainable_params / total_params:.2f}%)")

In [ ]:
# ============================================================
# CELL 3 — DATA PIPELINE
# Build a 3-item JSONL dataset in Alpaca format, apply the
# prompt template, EXPLICITLY tokenize, and strip all raw
# string columns before passing tensors to the trainer.
#
# WHY explicit tokenisation?
# SFTTrainer's auto dataset_text_field processing fails with
# "ValueError: too many dimensions 'str'" on recent trl versions.
# Pre-tokenising and removing string columns prevents this entirely.
# ============================================================

import json
import os
from datasets import load_dataset

# -----------------------------------------------------------
# 3a. Define the dummy training corpus
# Each record represents one Netezza SQL → PySpark translation pair.
# In production, replace these with your real labelled examples.
# -----------------------------------------------------------
RAW_DATA = [
    {
        "instruction": "Translate the following legacy Netezza SQL query into optimised PySpark DataFrame code.",
        "input": (
            "SELECT customer_id, SUM(order_total) AS total_spend\n"
            "FROM sales_fact\n"
            "WHERE order_date BETWEEN '2023-01-01' AND '2023-12-31'\n"
            "GROUP BY customer_id\n"
            "HAVING SUM(order_total) > 1000\n"
            "ORDER BY total_spend DESC;"
        ),
        "output": (
            "from pyspark.sql import functions as F\n\n"
            "result_df = (\n"
            "    spark.table('sales_fact')\n"
            "    .filter(F.col('order_date').between('2023-01-01', '2023-12-31'))\n"
            "    .groupBy('customer_id')\n"
            "    .agg(F.sum('order_total').alias('total_spend'))\n"
            "    .filter(F.col('total_spend') > 1000)\n"
            "    .orderBy(F.col('total_spend').desc())\n"
            ")"
        ),
    },
    {
        "instruction": "Translate the following legacy Netezza SQL query into optimised PySpark DataFrame code.",
        "input": (
            "SELECT p.product_name, c.category_name, COUNT(o.order_id) AS order_count\n"
            "FROM orders o\n"
            "JOIN products p ON o.product_id = p.product_id\n"
            "JOIN categories c ON p.category_id = c.category_id\n"
            "GROUP BY p.product_name, c.category_name\n"
            "ORDER BY order_count DESC\n"
            "LIMIT 10;"
        ),
        "output": (
            "from pyspark.sql import functions as F\n\n"
            "orders_df    = spark.table('orders')\n"
            "products_df  = spark.table('products')\n"
            "categories_df = spark.table('categories')\n\n"
            "result_df = (\n"
            "    orders_df\n"
            "    .join(products_df,  orders_df.product_id  == products_df.product_id,  'inner')\n"
            "    .join(categories_df, products_df.category_id == categories_df.category_id, 'inner')\n"
            "    .groupBy('product_name', 'category_name')\n"
            "    .agg(F.count('order_id').alias('order_count'))\n"
            "    .orderBy(F.col('order_count').desc())\n"
            "    .limit(10)\n"
            ")"
        ),
    },
    {
        "instruction": "Translate the following legacy Netezza SQL query into optimised PySpark DataFrame code.",
        "input": (
            "SELECT customer_id,\n"
            "       order_date,\n"
            "       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn\n"
            "FROM orders\n"
            "QUALIFY rn = 1;"
        ),
        "output": (
            "from pyspark.sql import functions as F, Window\n\n"
            "window_spec = Window.partitionBy('customer_id').orderBy(F.col('order_date').desc())\n\n"
            "result_df = (\n"
            "    spark.table('orders')\n"
            "    .withColumn('rn', F.row_number().over(window_spec))\n"
            "    .filter(F.col('rn') == 1)\n"
            "    .drop('rn')\n"
            ")"
        ),
    },
]

# -----------------------------------------------------------
# 3b. Persist to JSONL and load as a HuggingFace Dataset
# -----------------------------------------------------------
JSONL_PATH = "train.jsonl"
with open(JSONL_PATH, "w", encoding="utf-8") as fh:
    for record in RAW_DATA:
        fh.write(json.dumps(record) + "\n")

raw_dataset = load_dataset("json", data_files={"train": JSONL_PATH}, split="train")
print(f"Loaded {len(raw_dataset)} training examples")
print(f"Columns: {raw_dataset.column_names}")

# -----------------------------------------------------------
# 3c. Define the Alpaca-style prompt template
# The EOS token is appended so the model learns when to stop
# generating and does not hallucinate beyond the answer boundary.
# -----------------------------------------------------------
ALPACA_TEMPLATE = """\
Below is an instruction that describes a task, paired with an input that provides further context. \
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

EOS_TOKEN = tokenizer.eos_token  # Required: model must learn when to stop

def apply_template(batch):
    """Map raw Alpaca fields to a single formatted text string per sample."""
    texts = []
    for instruction, inp, output in zip(batch["instruction"], batch["input"], batch["output"]):
        text = ALPACA_TEMPLATE.format(
            instruction=instruction,
            input=inp,
            output=output,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

templated_dataset = raw_dataset.map(apply_template, batched=True)
print("\nSample prompt (first 500 chars):")
print(templated_dataset[0]["text"][:500])

# -----------------------------------------------------------
# 3d. Explicit tokenisation
# BUG PREVENTION: We tokenise here and return only numeric tensors
# (input_ids, attention_mask). This avoids the SFTTrainer bug where
# passing raw string columns causes "ValueError: too many dimensions 'str'".
# -----------------------------------------------------------
def tokenize_fn(batch):
    """Tokenise text into input_ids and attention_mask tensors."""
    encoded = tokenizer(
        batch["text"],
        truncation    = True,
        max_length    = MAX_SEQ_LENGTH,
        padding       = "max_length",   # Pad all samples to MAX_SEQ_LENGTH
        return_tensors = None,           # Return plain Python lists (datasets handles conversion)
    )
    return {
        "input_ids"      : encoded["input_ids"],
        "attention_mask" : encoded["attention_mask"],
    }

tokenized_dataset = templated_dataset.map(tokenize_fn, batched=True)

# -----------------------------------------------------------
# 3e. Drop ALL raw string columns — trainer must receive tensors only
# -----------------------------------------------------------
STRING_COLUMNS = ["instruction", "input", "output", "text"]
tokenized_dataset = tokenized_dataset.remove_columns(
    [col for col in STRING_COLUMNS if col in tokenized_dataset.column_names]
)
tokenized_dataset.set_format(type="torch")   # Return PyTorch tensors

# -----------------------------------------------------------
# Sanity check: only numeric tensor columns should remain
# -----------------------------------------------------------
print(f"\nFinal dataset columns : {tokenized_dataset.column_names}")
print(f"Dataset size          : {len(tokenized_dataset)} samples")
print(f"input_ids shape       : {tokenized_dataset[0]['input_ids'].shape}")
assert tokenized_dataset.column_names == ["input_ids", "attention_mask"], \
    "Unexpected columns remain — check remove_columns step"

In [ ]:
# ============================================================
# CELL 4 — TRAINING LOOP
# Initialise SFTTrainer with SFTConfig (NOT deprecated
# TrainingArguments) and kick off the fine-tuning run.
# ============================================================

# BUG PREVENTION: Import SFTConfig — not TrainingArguments.
# SFTConfig is the correct class for SFTTrainer from trl>=0.8.
from trl import SFTTrainer, SFTConfig

# -----------------------------------------------------------
# 4a. Training configuration
# Memory budget for T4 (15 GB VRAM):
#   Model weights (4-bit) : ~5 GB
#   Activations + grads   : ~7 GB  (batch=2, grad_accum=4)
#   Optimizer states      : ~2 GB  (adamw_8bit)
#   Buffer                : ~1 GB
# -----------------------------------------------------------
sft_config = SFTConfig(
    # -- Output & logging ---------------------------------
    output_dir              = "./outputs",
    logging_dir             = "./logs",
    logging_steps           = 10,
    report_to               = "none",          # Disable W&B / TensorBoard

    # -- Training schedule --------------------------------
    max_steps               = 60,              # Sufficient for a demo run
    warmup_steps            = 5,
    lr_scheduler_type       = "linear",
    learning_rate           = 2e-4,

    # -- Batch & gradient ---------------------------------
    per_device_train_batch_size  = 2,
    gradient_accumulation_steps  = 4,          # Effective batch = 2 * 4 = 8
    gradient_checkpointing       = True,

    # -- Precision ----------------------------------------
    # T4 does NOT support bfloat16 — must use float16.
    fp16                    = True,
    bf16                    = False,

    # -- Optimiser ----------------------------------------
    # adamw_8bit quantises the optimiser states to 8-bit,
    # cutting memory by ~50% vs standard AdamW.
    optim                   = "adamw_8bit",
    weight_decay            = 0.01,

    # -- Sequence length ----------------------------------
    # Must match the value used during tokenisation above.
    max_seq_length          = MAX_SEQ_LENGTH,

    # -- Dataset handling ---------------------------------
    # dataset_text_field is intentionally omitted — we
    # pre-tokenised the dataset and there are no string columns.
    dataset_num_proc        = 1,
)

# -----------------------------------------------------------
# 4b. Initialise the trainer
# BUG PREVENTION:
#   - processing_class=tokenizer  (NOT the deprecated tokenizer= argument)
#   - No dataset_text_field        (dataset is already tokenised)
# -----------------------------------------------------------
trainer = SFTTrainer(
    model              = model,
    train_dataset      = tokenized_dataset,
    processing_class   = tokenizer,    # Correct modern argument name
    args               = sft_config,
)

# -----------------------------------------------------------
# 4c. GPU memory snapshot before training
# -----------------------------------------------------------
gpu_stats = torch.cuda.get_device_properties(0)
reserved  = torch.cuda.max_memory_reserved() / 1e9
total_gb  = gpu_stats.total_memory / 1e9
print(f"GPU: {gpu_stats.name}  |  Total VRAM: {total_gb:.1f} GB  |  Reserved: {reserved:.2f} GB")

# -----------------------------------------------------------
# 4d. Train
# -----------------------------------------------------------
print("\nStarting training...")
train_result = trainer.train()

# -----------------------------------------------------------
# 4e. Report results
# -----------------------------------------------------------
peak_vram = torch.cuda.max_memory_reserved() / 1e9
metrics   = train_result.metrics

print("\n===== Training Complete =====")
print(f"Total steps    : {metrics.get('train_steps_per_second', 'N/A')}  steps/s")
print(f"Training loss  : {metrics.get('train_loss', 'N/A'):.4f}")
print(f"Peak VRAM used : {peak_vram:.2f} GB / {total_gb:.1f} GB")

In [ ]:
# ============================================================
# CELL 5 — EXPORT + SAVE TO GOOGLE DRIVE
# Save LoRA adapters, export GGUF, then persist both to Google
# Drive so they survive Colab runtime restarts.
#
# NOTE: Unsloth's save_pretrained_gguf automatically appends "_gguf"
# to whatever directory name you pass.
#   pass "model"  →  Unsloth creates "model_gguf/"   ✓
#   pass "model_gguf"  →  Unsloth creates "model_gguf_gguf/"  ✗
# ============================================================

import os
import shutil
import glob as glob_module
from google.colab import drive

LORA_SAVE_DIR  = "lora_model"   # Directory we write to directly
GGUF_DIR_ARG   = "model"        # Argument passed to save_pretrained_gguf
GGUF_SAVE_DIR  = "model_gguf"   # Actual directory Unsloth will create (appends _gguf)

# -----------------------------------------------------------
# 5a. Mount Google Drive
# Creates /content/drive/MyDrive/ — all files written here
# persist permanently across runtime restarts.
# -----------------------------------------------------------
print("Mounting Google Drive ...")
drive.mount("/content/drive")
DRIVE_BASE = "/content/drive/MyDrive/netezza-pyspark-model"
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive ready. Saving to: {DRIVE_BASE}/")

# -----------------------------------------------------------
# 5b. Save LoRA adapter weights + tokenizer (local first)
# Lightweight (~100 MB) — used by the inference notebook to
# reload the fine-tuned model without re-training.
# -----------------------------------------------------------
print(f"\nSaving LoRA adapters locally to ./{LORA_SAVE_DIR} ...")
model.save_pretrained(LORA_SAVE_DIR)
tokenizer.save_pretrained(LORA_SAVE_DIR)
print("LoRA adapters saved locally.")

# -----------------------------------------------------------
# 5c. Merge adapters into base model and export to GGUF (local)
# Quantisation method: q4_k_m
#   - 4-bit K-quant (medium)  — best quality/size tradeoff
#   - Expected output size    : ~4–5 GB
#   - Compatible with         : llama.cpp, Ollama, LM Studio, GPT4All
# -----------------------------------------------------------
print(f"\nMerging adapters and exporting GGUF (q4_k_m) ...")
print(f"Unsloth will write to ./{GGUF_SAVE_DIR}/ (appends _gguf automatically)")
print("This may take 5–10 minutes on a T4.")

model.save_pretrained_gguf(
    GGUF_DIR_ARG,
    tokenizer,
    quantization_method = "q4_k_m",
)

# Confirm GGUF output exists
if not os.path.isdir(GGUF_SAVE_DIR):
    raise FileNotFoundError(
        f"Expected '{GGUF_SAVE_DIR}/' not found. "
        f"Directories present: {os.listdir('.')}"
    )

gguf_files = glob_module.glob(os.path.join(GGUF_SAVE_DIR, "*.gguf"))
if gguf_files:
    for path in sorted(gguf_files):
        print(f"GGUF file : {path}  ({os.path.getsize(path)/1e9:.2f} GB)")
else:
    all_gguf = glob_module.glob("**/*.gguf", recursive=True)
    print(f"WARNING: No .gguf in ./{GGUF_SAVE_DIR}/  |  Found elsewhere: {all_gguf}")

# -----------------------------------------------------------
# 5d. Copy both outputs to Google Drive (persistent storage)
# After this step, /content/ can be wiped — your model is safe.
# -----------------------------------------------------------
DRIVE_LORA = os.path.join(DRIVE_BASE, LORA_SAVE_DIR)
DRIVE_GGUF = os.path.join(DRIVE_BASE, GGUF_SAVE_DIR)

print(f"\nCopying LoRA adapter to Drive: {DRIVE_LORA} ...")
if os.path.exists(DRIVE_LORA):
    shutil.rmtree(DRIVE_LORA)   # Remove stale copy if re-running
shutil.copytree(LORA_SAVE_DIR, DRIVE_LORA)
print("LoRA adapter copied to Drive.")

print(f"\nCopying GGUF model to Drive: {DRIVE_GGUF} ...")
print("This may take a few minutes (~4–5 GB transfer) ...")
if os.path.exists(DRIVE_GGUF):
    shutil.rmtree(DRIVE_GGUF)
shutil.copytree(GGUF_SAVE_DIR, DRIVE_GGUF)
print("GGUF model copied to Drive.")

# -----------------------------------------------------------
# Summary
# -----------------------------------------------------------
print("\n===== Export Complete =====")
print(f"LoRA adapter  (local) : /content/{LORA_SAVE_DIR}/")
print(f"GGUF model    (local) : /content/{GGUF_SAVE_DIR}/")
print(f"LoRA adapter  (Drive) : {DRIVE_LORA}/")
print(f"GGUF model    (Drive) : {DRIVE_GGUF}/")
print("\nDeploy with Ollama (after downloading GGUF from Drive):")
print("  ollama create netezza-pyspark -f <path-to>.Q4_K_M.gguf")
print("  ollama run netezza-pyspark")


In [ ]:
# ============================================================
# CELL 6 — INFERENCE & VALIDATION
# Test the fine-tuned model on unseen Netezza SQL queries.
# The model is already in memory from Cell 4 — no reload needed
# unless you restarted the runtime (see the fallback block below).
# ============================================================

from unsloth import FastLanguageModel
import torch

# -----------------------------------------------------------
# 6a. Runtime-restart fallback: reload model from LoRA adapter
# If you are running this cell fresh (runtime was restarted),
# uncomment the block below to reload from the saved adapter.
# Otherwise skip — the model from Cell 4 is already in memory.
# -----------------------------------------------------------
# RELOAD_FROM_DISK = True   # set True if runtime was restarted
# if RELOAD_FROM_DISK:
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name     = "lora_model",   # loads base + adapter together
#         max_seq_length = 2048,
#         dtype          = None,
#         load_in_4bit   = True,
#     )

# -----------------------------------------------------------
# 6b. Switch model to inference mode
# FastLanguageModel.for_inference() applies Unsloth's 2x-faster
# inference kernel and disables gradient tracking.
# MUST be called before generation — do not skip.
# -----------------------------------------------------------
FastLanguageModel.for_inference(model)

# -----------------------------------------------------------
# 6c. Define the inference prompt builder
# Uses the same Alpaca template as training, but leaves the
# ### Response: field empty — the model fills it in.
# -----------------------------------------------------------
INFERENCE_TEMPLATE = """\
Below is an instruction that describes a task, paired with an input that provides further context. \
Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Input:
{input}

### Response:
"""

def build_prompt(netezza_sql: str) -> str:
    return INFERENCE_TEMPLATE.format(
        instruction="Translate the following legacy Netezza SQL query into optimised PySpark DataFrame code.",
        input=netezza_sql.strip(),
    )

# -----------------------------------------------------------
# 6d. Test queries — all unseen during training
# -----------------------------------------------------------
TEST_QUERIES = [
    # Test 1: Simple aggregation with DISTINCT
    """\
SELECT department_id, COUNT(DISTINCT employee_id) AS headcount
FROM employee_fact
WHERE status = 'ACTIVE'
GROUP BY department_id
ORDER BY headcount DESC;""",

    # Test 2: Self-join pattern (manager hierarchy)
    """\
SELECT e.employee_id, e.name, m.name AS manager_name
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.employee_id
WHERE e.department_id = 10;""",

    # Test 3: Date truncation + running total (window function)
    """\
SELECT order_date,
       revenue,
       SUM(revenue) OVER (ORDER BY order_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
FROM daily_revenue
ORDER BY order_date;""",
]

# -----------------------------------------------------------
# 6e. Run inference on each test query
# max_new_tokens=512 is enough for most PySpark translations.
# temperature=0.1 keeps output deterministic and code-like.
# use_cache=True is required for Unsloth's fast inference path.
# -----------------------------------------------------------
print("=" * 65)
print("  INFERENCE VALIDATION — Netezza SQL → PySpark Translator")
print("=" * 65)

for idx, sql in enumerate(TEST_QUERIES, start=1):
    prompt = build_prompt(sql)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens  = 512,
            temperature     = 0.1,
            do_sample       = True,
            use_cache       = True,
            pad_token_id    = tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print(f"\n{'─' * 65}")
    print(f"  TEST {idx}")
    print(f"{'─' * 65}")
    print("INPUT (Netezza SQL):")
    print(sql)
    print("\nOUTPUT (PySpark):")
    print(response)

print(f"\n{'=' * 65}")
print("  Validation complete.")
print("=" * 65)
